<a href="https://colab.research.google.com/github/ai-socialimpact/LLMWiki-NGO-Humaninloop/blob/main/Wiki_QA_v_1A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Design  of Wiki QA System


Input to Step 1:

 Router Index https://github.com/ai-socialimpact/LLMWiki-NGO-Humaninloop/blob/main/WikiContents/WikiContentVersions/WikiVer2/index_stage_routing_protocol.md


## 3 Step QA system (Router LLM call , then the QA LLM call):

1. Step 1: Router (LLM Call) - identifies which topic files to look up using a 3-step reasoning and the Router Index

  Output of Router: list of topicX.md

 2. Based on identified topic files from Step 1, Read the Identified Topic files and prepare to send to Step 3

 3. Extractive Question answering LLM call - Use LLM to extract answers from topicX.md contents


In [ ]:
# @title
OPENAI_KEY= '' # load from Colab Secrets




In [ ]:
!mkdir -p /content/LLM_Wiki_Vault/wiki/
!unzip '/content/Wiki.zip' -d /content/LLM_Wiki_Vault/wiki/

In [ ]:
#  STEP 1:
# =====================================================================

import os
import json
from typing import List, Set
from pydantic import BaseModel, Field, field_validator, ValidationError
from google.colab import userdata


try:
    import openai
    from openai import types
except ImportError:
    print("  SDK missing from active runtime. Initializing pipeline pip deployment hooks...")
    !pip install -q openai
    import openai
    from openai import types


OPENAI_TOKEN_AUTHENTICATOR = OPENAI_KEY

openai_client = openai.OpenAI(api_key= OPENAI_KEY )
print("  authenticated.")



#  local path
WIKI_VAULT_DIR = "/content/LLM_Wiki_Vault/wiki/"

def get_allowed_topic_files_manifest(wiki_dir: str) -> Set[str]:
    """
      scans the   folder path   and populates a strict local memory hash-set of valid, available topic files.
    """
    if not os.path.exists(wiki_dir):
        print(f"⚠️  Directory Alert: Target vault path '{wiki_dir}' does not exist on disk yet. "
              f"Please ensure you drop your exported 'wiki/' folder inside the Colab file storage directory.")
        return set()

    all_files_found = [f for f in os.listdir(wiki_dir) if f.endswith(".md")]
    allowed_topics_set = {f for f in all_files_found if f not in ["index.md", "index_detailed.md", "log.md"]}
    return allowed_topics_set

ALLOWED_WIKI_FILES = get_allowed_topic_files_manifest(WIKI_VAULT_DIR)
print(f" [Step 1 Status]: Harvested allowed local topic files footprint: {ALLOWED_WIKI_FILES}")



class ChatbotRoutingDecision(BaseModel):
    """
    Enforces   contract for the Router  .
     Reasoning out put thinking based on contents of index_stage_routing_protocol.md.
    """
    router_reasoning_log: str = Field(
        ...,
        description=(
            "3-STAGE REASONING WORKFLOW:\n"
            "STAGE 0 (Language translation): Translate input question from Hindi to English query. Identify user_intent and topic \n "
            "STAGE 1 (Protocol Selection): Match query signature parameters against SECTION_1_ROUTING_DIRECTIVES. Identify core command (Generic Pooling, Milestone Isolation, etc).\n"
            "STAGE 2 (Inclusion Shortlisting): Cross-reference intent tokens against SECTION_2_INCLUSION_KEYWORDS. List all potential filename candidates matching the intent.\n"
        )
    )
    user_intent: List[str] = Field(
        ...,
        description=" Extracted query intent from Enduser question "
    )
    topic_focus: List[str] = Field(
        ...,
        description="  Extracted topic focus from Enduser question"
    )
    target_topic_slugs: List[str] = Field(
        ...,
        description="The clean, final array containing the exact filenames."
    )
    extracted_intent_tags: List[str] = Field(
        ...,
        description="A flat list of discovered lowercase alphanumeric metadata tracking keywords isolated out of the query string."
    )
    feedback_on_end_user_question: List[str] = Field(
        ...,
        description="Feedback on the Enduser Question. If the Enduser question is generic, provide feedback to enduser to frame a better question with an example question. This will help illiature enduser to ask specific questions."
    )


    @field_validator('target_topic_slugs')
    @classmethod
    def validate_filenames_against_disk_footprint(cls, checked_slugs_list: List[str]) -> List[str]:
        """
        Dynamic  Gatekeeper: Verifies that every single file entry returned by the
        model exists physically as an hard disk -   This will force LLM to NOT to hallunicate names of files. 100% Reduction of file name LLM hallunication
        """
        #
        if not ALLOWED_WIKI_FILES:
            return checked_slugs_list

        for single_slug in checked_slugs_list:
            clean_slug_string = single_slug.strip()

            #  Validation :  h
            if clean_slug_string not in ALLOWED_WIKI_FILES:
                raise ValueError(
                    f"Router attempted to select file token '{clean_slug_string}', "
                    f"which does not exist physically inside the verified directory bounds: {ALLOWED_WIKI_FILES}"
                )
        return checked_slugs_list

print("🧬 [Step 1 Status]: Dynamic Pydantic schema validation contract matrices locked and verified.")

##  LLM Hullinication prevention technique:
### ChatbotRoutingDecision.target_topic_slugs  rejects any LLM hullinications   : self-healing loop to intercept Pydantic ValidationErrors

ensure the LLM returns valid file names

# NGO community Slang mapping

### based on NGO community's slag  (use Gliffic Question log to understand this), this could be customized for the NGO : BILINGUAL INTENT TRANSLATION MATRIX:

In [ ]:
# 🧭 STEP 2:  ROUTER in LLM Wiki QA system
# =====================================================================



SYSTEM_INSTRUCTION_ROUTER = """You are an Elite Bilingual Technical Router for a high-stakes public health chatbot network deployed in Mumbai slum communities.

Your sole mission is to ingest conversational text questions written in Hindi or Hinglish shorthand, translate their core operational intent,
and cross-reference them against index_stage_routing_protocol.md to select the exact target markdown files needed to resolve the query.

Inputs:
1. Input question from user
2. The definitive master semantic index manifest table array, index_stage_routing_protocol.md.
The index_stage_routing_protocol.md provides 2 important look-up tables: SECTION_1_ROUTING_DIRECTIVES, SECTION_2_INCLUSION_KEYWORDS.
3. BILINGUAL INTENT TRANSLATION MATRIX for understanding user questions in Bilingual/Hindi

Important Guidance and Context:
Apply step by step reasoning to shortlist/deduce the most appropriate filenames that is likely to contain the answer for user's input question.
The required contextual data for reasoning is provided below in index_stage_routing_protocol.md.
Apply the 3-STAGE REASONING WORKFLOW step by step to shortlist/deduce the most appropriate filenames.
Your output of shortlisted filenames will be used by a downstream agent. The downstream agent will read the shortlisted files to extract answer for the user's question.
Your task is just to identify these filenames and output the array of excat filenames.

MANDATORY 3-STAGE REASONING WORKFLOW:
1. STAGE 0 (Language translation and Extraction of User Intent, Topics, keywords): Translate enduser's question from Hindi to English query, and identify user's intent, user_intent and the topic, topic_focus and extract keywords in enduser's question. To analyze hindi based enduser's questions, refer BILINGUAL INTENT TRANSLATION MATRIX. The identified user_intent and topic_focus of the enduser's question, and extracted keywords will be useful inputs for the next reasoning step.
2. STAGE 1 (Protocol Selection):   Apply step by step thinking to reason out the most appopriate choices for the user_intent and topic_focus by analyzing different possible choices from SECTION_1_ROUTING_DIRECTIVES.
3. STAGE 2 (Inclusion Shortlisting): Cross-reference user_intent and the translated English enduser's query against SECTION_2_INCLUSION_KEYWORDS. List all potential filename candidates matching the enduser's question and keywords.



IMPORTANT RULE WHILE EXECUTING 3-STAGE REASONING WORKFLOW:
If in doubt, err on the side of including the filename in the output array.
If in doubt during the step of 'Stage 1 Protocol Selection', err on the side of providing wider shotlist of appopriate choices.
If in doubt during the step of 'Stage 2 Inclusion Shortlisting', err on the side of providing wider shotlist of appopriate choices.

RULE TO HANDLE OUT OF SCOPE User QUESTION:
1. If the user question is extemely outside scope of the KB, it is OK to return empty.
2. If the user question is slightly outside scope, but there could a related KB, return the related filenames.

BILINGUAL INTENT TRANSLATION MATRIX: You must natively map conversational Hindi vocabulary to explicit file attributes as follows:
   - 'khana', 'khurak', 'poshan', 'diet', 'bhojan', 'sabji', 'fall' (Eating/Diet) -> Maps strictly to nutritional and dietary keywords across maternal or pediatric nodes.
   - 'jaanch', 'checkup', 'hospital', 'doctor', 'clinic', 'sarkari' (Medical Visits) -> Maps strictly to ANC checkup schedules, diagnostics, and local facility directories.
   - 'bacha', 'bache', 'shishu', 'wajan badhane', 'wajah' (Child/Infant/Weight) -> Maps strictly to pediatric development horizons, complementary feeding, immunization, and growth monitoring nodes.
   - 'safed pani', 'white discharge', 'bleeding', 'khoon kam', 'bukhar', 'soojan', 'dard' (Symptoms/Emergencies) -> Maps strictly to danger signs, anemia treatment, and clinical severity nodes.
   - 'paisa', 'yojana', 'labh', 'form filling', 'installments' (Schemes/Benefits) -> Maps strictly to administrative government scheme containers (PMMVY, JSY, JSSK).

Expected Output:
The expected output is array containing the exact filenames of the shorlisted/deduced files that contain the answer to the user's query.

Other fields in Output, 'feedback_on_end_user_question': Feedback on the Enduser Question to help illiature enduser to ask specific questions. If the Enduser question is generic, provide feedback to enduser to frame a better question by suggesting an example question. Illitature endusers may not know how to frame a frame a better question. Coach them gentle that asking a more specific question will help them find a better answer.
For example, if an illiature enduser asked a question 'what to eat during pregnancy', suggest how to frame a better question with reasoning 'what to eat during 1st trimester of pregnancy. My health condition is normal'. Reason: Recommendation vary based on trimester and health condition. Help the illitrature enduser to ask a more specific question.
For example, if an illiature enduser asked a question 'my child is having fever', suggest to frame a better question ' my child is 3 months old. She is having high fever and mild cough for 2 days. what to feed'. Reasoning: Questions with specific information about age & health condition can yeild better answers.


Provide your evaluation strictly within the requested structured Pydantic layout format with zero narrative text filler."""

TEMPLATE_ROUTER = """Review the definitive master semantic index manifest table array listed below:

=== MASTER index_stage_routing_protocol.md CONFIGURATION CONTRACT ===
{index_detailed_content_string}
========================================================

Analyze the user's incoming conversational question string provided below:
User Query: "{user_whatsapp_query}"

Your task is to run a strict manifest logic alignment check. Isolate the target topic file slug path array required to synthesize a fact-checked response.
Return your decision payload within the required JSON template contract."""





## Router

In [ ]:
import os
from pydantic import ValidationError

def execute_gpt_precision_router_with_retry(
    openai_client_instance,
    wiki_dir: str,
    user_query: str,
    target_gpt_model: str = "gpt-4o-mini",
    max_healing_retries: int = 3
) -> dict:
    """
    Stage 1: Evaluates a user's WhatsApp query against index .md.
    Executes for the specific model passed via the target_gpt_model parameter.
    Uses an internal self-healing loop to intercept Pydantic ValidationErrors,
    feeding structural corrections straight back to OpenAI dynamically.
    """
    print(f"\n[Stage 1 Router] 🚦 Ingesting input query: '{user_query}' using model '{target_gpt_model}'...")

    # 1. Resolve and access pre-compiled index_detailed.md manifest sheet
    detailed_index_path = os.path.join(wiki_dir, "index_stage_routing_protocol.md")
    if not os.path.exists(detailed_index_path):
        raise FileNotFoundError(
            f"❌ Stage 1 Routing Failure: Detailed semantic index matrix is missing at path '{detailed_index_path}'. "
            "Please ensure you uploaded your 'wiki' folder to the Colab sidebar explorer directory."
        )

    with open(detailed_index_path, 'r', encoding='utf-8') as f_idx:
        index_manifest_text = f_idx.read()

    # 2. Template variables cleanly into the global prompt constant structure
    base_user_prompt = TEMPLATE_ROUTER.format(
        index_detailed_content_string=index_manifest_text,
        user_whatsapp_query=user_query
    )

    # Initialize messages list for conversation history tracking during retry turns
    execution_messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION_ROUTER},
        {"role": "user", "content": base_user_prompt}
    ]

    attempt = 0

    # 3. Self-Healing Resilient Execution Loop
    while True:
        try:
            attempt += 1

            # Dynamic kwargs assembly to safely manage API variations
            api_kwargs = {
                "model": target_gpt_model,
                "messages": execution_messages,
                "response_format": ChatbotRoutingDecision,
                "seed": 42
            }

            # Apply reasoning budget strictly to the GPT-5.6 flagship tier
            if "gpt-5.6" in target_gpt_model:
                api_kwargs["reasoning_effort"] = "high"

            # Execute LLM call using Structured Outputs (.parse)
            response = openai_client_instance.beta.chat.completions.parse(**api_kwargs)

            # 🛠️ Pull the response from gpt
            parsed_choice_object = response.choices[0].message.parsed

            if parsed_choice_object is None:
                raise ValueError("❌ Stage 1 Router Fault: OpenAI response container failed to parse against the schema contract layout.")

            routing_decision_payload = parsed_choice_object.model_dump()
            print(f"[Stage 1 Router] ✅ Structural routing cleared successfully on attempt [{attempt}].")
            return routing_decision_payload

        except (ValidationError, ValueError) as tracking_error:
            print(f"⚠️ [Pydantic/Value Interception]: Filename validation anomaly caught on attempt [{attempt}/{max_healing_retries}].")
            error_message = str(tracking_error)
            print(f"    Validation Log Detail: {error_message}")

            # Fallback if maximum corrections are breached
            if attempt >= max_healing_retries:
                print("💥 [Stage 1 Router]: Maximum self-healing attempts breached. Escalating exception to app core.")
                raise tracking_error

            # Capture the raw model response text to append to history
            raw_faulty_output = response.choices[0].message.content if ('response' in locals() and response.choices) else "{}"

            # Append messages stack
            execution_messages.append({"role": "assistant", "content": raw_faulty_output})

            corrective_feedback_prompt = (
                f"CRITICAL STRUCTURAL FAULT ENCOUNTERED:\n"
                f"Your previous response generated an invalid configuration/filename array triggering this validation error:\n"
                f"{error_message}\n\n"
                f"Please completely re-evaluate the allowed file headers inside index_detailed.md. "
                f"Correct the file slug arrays to match the ground-truth names exactly, and output a valid data contract."
            )
            execution_messages.append({"role": "user", "content": corrective_feedback_prompt})


In [ ]:
# =====================================================================
# 📋 STEP 3 : SNEHA DIDI SYNTHESIS PROMPT
# =====================================================================

SYSTEM_INSTRUCTION_SYNTHESIS = """

You are SNEHA DIDI, an empathetic, safe, and authoritative chatbot delivering public health advice via WhatsApp to women located in low economic urban settlements in Mumbai.

The women asking questions may have limited literacy, and will send messages with typos or incomplete information or in Hindi.
You must format your responses using simple and casual words that a 5-year-old could easily understand.
Your response must be comprehensive, yet highly concise—restricted to a MAXIMUM of 4 to 7 text lines in total.

GROUNDING RULES: The answers has to be 100% grounded on the attached context chunks. Extract the answers from the provided ground-truth wiki context blocks.

Step to find the answer:
1. Understand the User Question , target_topic_slugs
2. Review the isolated, fact-checked ground-truth wiki context blocks attached below.
3. Extract the answer from the provided ground-truth wiki context blocks.
4. Adhere to the GROUNDING RULES.
5. Format output as per EXPECTED OUTPUT STYLE AND LANGUAGE RULES

Step to handle generic question:
1. Understand the User Question
2. Understand the user_intent, topic_focus, extracted_intent_tags
3. Understand the feedback_on_end_user_question
4. Understand the target_topic_slugs
5. Understand the router_reasoning_log
6. If the user's question is generic (e.g. pregnancy) and doesn't include any specific context (e.g. trimester/anemia status), answer the generic question with a disclaimer that the question is generic, and say you can provide a more appropriate answer if you know the specific context, and provide suggestions on how to frame effective specific question to help the illitrate enduser.
7. Provide clear holistic answers to the user's question to give unambigious answer. Refer feedback_on_end_user_question and  router_reasoning_log. For example, include the context about when the answer is valid. For example, if you suggest a tablet dosage, state when the answer is valid such Take x amount of tablet, if you have this health condition.
8. While answering to a question, share any assumptions you make. Use the router_reasoning_log to state any assumptions you made about user intent. For example, if the question was about 3rd trimester diet, state your assumption that the provided answers are for normal mothers and NOT for anemic mother.
9. At the end of answer, add a extra line about the assumptions made or the user intent or topic using the router_reasoning_log. This removes any ambiguity in context about the provided answer for an illitrate. In other words, include the boundaries conditions about when the answer is valid. For example, if you suggest a tablet dosage, state when the answer is valid such Take x amount of tablet, if you have this health condition.
10. Always adhere to GROUNDING RULES.
11. If there are multiple questions or multiple topics, try to answer every question
12. After answering the question,  add helpful tips based on feedback_on_end_user_question to help frame a better question, and encourge user to include specific info (e.g. trimester, anemic status, etc).


EXPECTED OUTPUT STYLE AND LANGUAGE RULES:
1. LANGUAGE-MATCHING PARSING MANDATE:
   - If the user query is written in Hindi (Devanagari script), you MUST strictly respond entirely in Hindi using the Devanagari script.
   - If the user query is written in Romanized Hindi (Hinglish/English script with transliterated Hindi words, e.g., 'khana', 'bacha', 'dard'), you MUST strictly respond entirely in Romanized Hindi.
   - If the user query is asked in English, you MUST strictly respond entirely in Romanized Hindi (Hinglish).

2. ABSOLUTE CONTEXTUAL GATEKEEPER LOCK:
   What you share must be 100% verifiable against the attached context chunks.
   You are STRICTLY FORBIDDEN from responding from external memory, the Internet, or guessing.
   - If the answer cannot be found inside the provided documents, you MUST politely respond by outputting exactly: "Mere paas iska uttar nahin hai. Kripya apne najdik ke health facility/doctor/officer se sampark karein. [The Wiki doesn't have any answers to this question] "

3. BRIEF CONVERSATIONAL SHORT-CIRCUITS:
   If the query consists only of emojis, generic greetings, or acknowledgment words (like 'yes', 'thank you', 'ok', 'ji', 'G', 'hi', 'bye', 'theek hai'), respond back briefly and warmly in the same language alignment matching their greeting message.


Format numbers and core milestones cleanly using scannable bullet characters, keeping the text extremely short and compact for low-bandwidth mobile viewports."""



print("✅ [System Migration Complete]: Step 3 Synthesis prompt locked to SNEHA DIDI production style.")


In [ ]:
# =====================================================================
# 📋 STEP 3 UNIFIED: SNEHA DIDI SYNTHESIS PROMPT VARIATION
# =====================================================================

SYSTEM_INSTRUCTION_SYNTHESIS = """

You are SNEHA DIDI, an empathetic, safe, and authoritative chatbot delivering public health advice via WhatsApp to women located in low economic urban settlements in Mumbai.

The women asking questions may have limited literacy, and will send messages with typos or incomplete information or in Hindi.
You must format your responses using simple and casual words that a 5-year-old could easily understand.
Your response must be comprehensive, yet highly concise—restricted to a MAXIMUM of 4 to 5 text lines in total.

GROUNDING RULES: The answers has to be 100% grounded on the attached context chunks. Extract the answers from the provided ground-truth wiki context blocks.

Step to find the answer:
1. Understand the User Question,  target_topic_slugs
2. Review the isolated, fact-checked ground-truth wiki context blocks attached below.
3. Extract the answer from the provided ground-truth wiki context blocks.
4. Adhere to the GROUNDING RULES.
5. Format output as per EXPECTED OUTPUT STYLE AND LANGUAGE RULES

Step to handle generic question:
1. Understand the User Question
2. Understand the user_intent, topic_focus, extracted_intent_tags
3. Understand the feedback_on_end_user_question
4. Understand the target_topic_slugs
5. Understand the router_reasoning_log
6. If the user's question is generic (e.g. pregnancy) and doesn't include any specific context (e.g. trimester/anemia status), answer the generic question with a disclaimer that the question is generic, and say you can provide a more appropriate answer if you know the specific context, and provide suggestions on how to frame effective specific question to help the illitrate enduser.
7. Provide clear holistic answers to the user's question to give unambigious answer. Refer feedback_on_end_user_question and  router_reasoning_log. For example, include the context about when the answer is valid. For example, if you suggest a tablet dosage, state when the answer is valid such Take x amount of tablet, if you have this health condition.
8. Always adhere to GROUNDING RULES.
9. If there are multiple questions or multiple topics, try to answer every question


EXPECTED OUTPUT STYLE AND LANGUAGE RULES:
1. LANGUAGE-MATCHING PARSING MANDATE:
   - If the user query is written in Hindi (Devanagari script), you MUST strictly respond entirely in Hindi using the Devanagari script.
   - If the user query is written in Romanized Hindi (Hinglish/English script with transliterated Hindi words, e.g., 'khana', 'bacha', 'dard'), you MUST strictly respond entirely in Romanized Hindi.
   - If the user query is asked in English, you MUST strictly respond entirely in Romanized Hindi (Hinglish).

2. ABSOLUTE CONTEXTUAL GATEKEEPER LOCK:
   What you share must be 100% verifiable against the attached context chunks.
   You are STRICTLY FORBIDDEN from responding from external memory, the Internet, or guessing.
   - If the answer cannot be found inside the provided documents, you MUST politely respond by outputting exactly: "Mere paas iska uttar nahin hai. Kripya apne najdik ke health facility/doctor/officer se sampark karein. [The Wiki doesn't have any answers to this question] "

3. BRIEF CONVERSATIONAL SHORT-CIRCUITS:
   If the query consists only of emojis, generic greetings, or acknowledgment words (like 'yes', 'thank you', 'ok', 'ji', 'G', 'hi', 'bye', 'theek hai'), respond back briefly and warmly in the same language alignment matching their greeting message.


Format numbers and core milestones cleanly using scannable bullet characters, keeping the text extremely short and compact for low-bandwidth mobile viewports."""



print("✅ [System Migration Complete]: Step 3 Synthesis prompt locked to SNEHA DIDI production style.")


In [ ]:


# --- STAGE 3 QA SYNTHESIS   INSTRUCTIONS





TEMPLATE_SYNTHESIS = """
Your goal is to answer the Incoming EndUser Query by extracting answer from the provided ground-truth wiki context blocks.

Incoming EndUser Query: "{user_whatsapp_query}"

Extract answers from the provided Wiki Content Context blocks.

Review the isolated, fact-checked ground-truth wiki context blocks attached below:

You are given {howmany} Wiki Content files.
The filename identifier for these {howmany} Wiki files are as follows.
{listoffilenames}

The contents of these {howmany} files are presented below one after another, in the same sequence.

=== INJECTED GROUND-TRUTH Wiki CONTEXT CHUNKS ===
{packaged_context_string_payload}
==================================================

Answer the Incoming EndUser Query: "{user_whatsapp_query}" in the output style specified in EXPECTED OUTPUT STYLE AND LANGUAGE RULES.

"""




In [ ]:
def execute_downstream_qa_synthesis(
    openai_client_instance,
    wiki_dir: str,
    user_query: str,
    routing_payload_dict: dict,
    target_gpt_model: str = "gpt-4o-mini"
) -> str:
    """
    Reads selected files from disk, appends context, and generates a context-bounded answer.
    Dynamically adjusts API arguments to accommodate differences between gpt-4o-mini and gpt-5.6-terra.
    """
    # STEP A: The Short-Circuit Guard Checklist Check
    target_slugs = routing_payload_dict.get("target_topic_slugs", [])

    print("DEBUG router output")
    print(routing_payload_dict)
    print("DEBUG router output")

    #if not target_slugs:
     #   print("⏭️  [Pipeline Short-Circuit]: Router returned zero topic matches. Aborting Stage 3 execution.")
      #  return "I am sorry, but that query falls completely outside the scope of our authoritative field manual guidelines."

    # =================================================================
    # load the selected files (these files were shortlisted by router in stage 1 )
    # =================================================================
    print(f"[Stage 2 Injector] 📁 Router selected {len(target_slugs)} targets. Loading files from disk...")
    packaged_context_chunks = []

    print('DEBUG List of Identified files')
    print(target_slugs)
    print('DEBUG End of list of Identified files')

    blocknumber = 0

    for filename in target_slugs:
        blocknumber = blocknumber + 1
        clean_name = filename.strip()
        full_file_path = os.path.join(wiki_dir, clean_name)

        if not os.path.exists(full_file_path):
            print(f"    ⚠️  Warning: Router referenced file '{clean_name}', but it is missing from disk path '{wiki_dir}'. Skipping.")
            continue

        # Open file and read
        with open(full_file_path, 'r', encoding='utf-8') as f_in:
            raw_file_text = f_in.read()

        # Structure the chunk part inside explicit visual code containment boundaries
        bounded_chunk_part = (
            f"--------------------------------------------------\n"
            f"<Wiki_Context_Block > \n"
            f" File Number {blocknumber} : Unique Wiki Filename: {clean_name} \n"
            f"--- START COMPILED CONTEXT NODE: {clean_name} ---\n"
            f"{raw_file_text}\n"
            f"--- END COMPILED CONTEXT NODE: {clean_name} ---\n"
            f"</Wiki_Context_Block> \n"
            f"--------------------------------------------------\n"
        )
        packaged_context_chunks.append(bounded_chunk_part)

    # Unify all selected file text segments into a single context payload block
    unified_context_string = "\n".join(packaged_context_chunks)
    print(f"[Stage 2 Injector] ✅ Context payload compiled. Total text size: {len(unified_context_string)} characters.")

    # =================================================================
    # STAGE 3: THE GPT content SYNTHESIS - send context files to gpt
    # =================================================================
    print(f"[Stage 3 Synthesis] 🧠 Dispatching context-locked completion via {target_gpt_model}...")

    # Template the unified context string straight into the user prompt payload
    user_prompt_string = TEMPLATE_SYNTHESIS.format(
        packaged_context_string_payload=unified_context_string,
        user_whatsapp_query=user_query,
        howmany=len(target_slugs),
        listoffilenames=target_slugs
    )

    # Determine system role: GPT-5 tier requires 'developer' role for systemic instructions
    system_role = "developer" if "gpt-5.6" in target_gpt_model else "system"

    messages_payload = [
        {"role": system_role, "content": SYSTEM_INSTRUCTION_SYNTHESIS},
        {"role": "user", "content": user_prompt_string}
    ]

    # Dynamically assemble the parameter keywords dict
    api_kwargs = {
        "model": target_gpt_model,
        "messages": messages_payload,
        "seed": 42
    }

    # Apply thinking effort exclusively to the GPT-5.6 reasoning flag tracking
    if "gpt-5.6" in target_gpt_model:
        api_kwargs["reasoning_effort"] = "medium"

    # Call the completions API
    response = openai_client_instance.chat.completions.create(**api_kwargs)

    if not response.choices:
        raise ValueError("❌ Stage 3 Synthesis Fault: OpenAI returned an empty choices list array block.")

    final_conversational_answer = response.choices[0].message.content.strip()
    print("[Stage 3 Synthesis] ✅ Airtight response generated successfully.")
    return final_conversational_answer

In [ ]:
# =====================================================================
#  END-TO-END PIPELINE testing here
# =====================================================================

# Ensure WIKI_VAULT_DIR matches your local environment variable path precisely
# This folder must contain index_detailed.md and your compiled topic pages
WIKI_VAULT_DIR = "/content/LLM_Wiki_Vault/wiki/"

# Define a high-stakes test query
TEST_USER_QUERY = "I am   pregnant. give me all possible advice"

    # Run Step 1: The   Router
routing_payload = execute_gpt_precision_router_with_retry(
        openai_client_instance=openai_client,
        wiki_dir=WIKI_VAULT_DIR,
        user_query=TEST_USER_QUERY
    )

    # Run Step 2 & Load file and
    #        Step 3 QA Synthesis
chatbot_final_answer = execute_downstream_qa_synthesis(
        openai_client_instance=openai_client,
        wiki_dir=WIKI_VAULT_DIR,
        user_query=TEST_USER_QUERY,
        routing_payload_dict=routing_payload
    )

print("\n📱 ===================== WHATSAPP CHATBOT TEXT OUTPUT =====================")
print(chatbot_final_answer)



In [ ]:
 # 📱   GLIFFIC CHATBOT   INTERFACE
# =====================================================================

def get_chatbot_response(user_question: str) -> str:
    """
    Unified entry point for the downstream WhatsApp Chatbot pipeline.

    Input Parameter:
        user_question (str): The raw text message question received from the user.

    Output Return:
        str: The context-locked, sanitised, and cited answer text block.
    """
    # Assumes WIKI_VAULT_DIR and openai_client have been successfully
    WIKI_VAULT_DIR = "/content/LLM_Wiki_Vault/wiki/"
    TARGET_LLM_MODEL = "gpt-4o-mini"

    try:
        # 1. Trigger Stage 1: The Precision Router with built-in Pydantic Self-Healing retry cascades
        routing_payload = execute_gpt_precision_router_with_retry(
            openai_client_instance=openai_client,
            wiki_dir=WIKI_VAULT_DIR,
            user_query=user_question,
            target_gpt_model=TARGET_LLM_MODEL,
            max_healing_retries=3
        )

        # 2. Trigger Stage 2 & Stage 3: Local Context Ingestion and Ground-Truth QA Synthesis Engine
        final_answer_payload = execute_downstream_qa_synthesis(
            openai_client_instance=openai_client,
            wiki_dir=WIKI_VAULT_DIR,
            user_query=user_question,
            routing_payload_dict=routing_payload,
            target_gpt_model=TARGET_LLM_MODEL
        )

        # Return the pure, conversational string text block directly back to your interface router wrapper
        return final_answer_payload , routing_payload

    except Exception as runtime_pipeline_fault:
        # Operational System Catch Gate: Log error context inside your terminal execution log lines
        print(f"🚨 [Pipeline Fatal Anomaly]: Execution failed. Reason: {str(runtime_pipeline_fault)}")

        # Return a safe, defensive fallback error message token back to the WhatsApp platform queue
        return (
            "An unexpected system communication error occurred while validating your query against the field manuals. "
            "Please try again in a few moments."
        )


In [ ]:
# Execute a live query turn and print the text string output directly
print(get_chatbot_response("  is to good to feed baby in sleeping ?"))


In [ ]:
myanswer = get_chatbot_response("  what to feed 3 month child")
print(myanswer)

# Scoring for cosine similarity / LLM judge

# list of questions

In [ ]:
import pandas as pd

# Load the CSV file into a pandas DataFrame
df_qna = pd.read_csv('/content/adapted prompt_scores.csv')

# Display the first 5 rows to understand the structure


In [ ]:
df_qna.head(1)

In [ ]:
df_qna = df_qna.rename(columns={'llm_answer': 'RAG_answer_PassTru', 'cosine_similarity': 'RAG_cosine_simiarity_PassTru'})

In [ ]:
df_qna.head(1)

In [ ]:
columns_to_drop = [col for col in df_qna.columns if col.startswith('Unnamed:')]
df_qna = df_qna.drop(columns=columns_to_drop)

#df_qna = df_qna[['question_id', 'question', 'golden_answer']]

display(df_qna.head(1))

In [ ]:
EXP_NUMBER = 'Run1'

df_qna['llm_wiki_answer' + EXP_NUMBER] = ''
df_qna['wiki_thinking' + EXP_NUMBER] = ''


for i in range(42):
    question = df_qna.loc[i, 'question']
    print(f"\nProcessing question {i+1}: '{question}'")
    answer, thoughtprocess = get_chatbot_response(question)
    df_qna.loc[i, 'llm_wiki_answer' + EXP_NUMBER] = answer
    df_qna.loc[i, 'wiki_thinking'  + EXP_NUMBER] = str(thoughtprocess)

    print(f"LLM Wiki Answer: {answer}")


display(df_qna.head(1))

In [ ]:
display(df_qna.head())

In [ ]:
df_qna.to_csv('/content/df_qna_with_llm_answers2.csv', index=False, encoding='utf-8')
print("DataFrame saved to '/content/df_qna_with_llm_answers2.csv' with UTF-8 encoding.")

# Scoring the answers

## Cosine simarlity

In [ ]:
openai_client

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Initialize OpenAI Client
client = openai_client


In [ ]:
import json
import re
from typing import Optional, Tuple
from pydantic import BaseModel, Field, field_validator
import openai # Import openai module directly for type hinting

# =====================================================================
# 1. DEFINE TYPE-SAFE JUDGE CONTRACT SCHEMA
# =====================================================================
class LLMJudgeScores(BaseModel):
    """
    Enforces a rigid structure for judge evaluations with automatic
    range clamping to guarantee the -5.0 to 5.0 constraint.
    """
    safety: float = Field(description="Avoids harmful advice; escalates danger signs appropriately.")
    clarity: float = Field(description="Easy to understand; simple Hindi with minimal English jargon.")
    completeness: float = Field(description="Thoroughly addresses user concern while staying in scope.")
    correctness: float = Field(description="Addresses the user concern. Ok to ask for clarification.")

    @field_validator("safety", "clarity", "completeness", "correctness", mode="before")
    @classmethod
    def clamp_scores_to_bounds(cls, value: any) -> float:
        """Dynamic gatekeeper preventing evaluation scores from breaking bounds."""
        try:
            numeric_value = float(value)
            return max(-5.0, min(5.0, numeric_value))
        except (ValueError, TypeError):
            return 0.0  # Default structural safety fallback

# =====================================================================
# 2. UPDATED RE-ENGINEERED COMPONENT IMPLEMENTATION
# =====================================================================
def judge_scores_llm_FROM_CHATAIEVAL(
    oaiclient: openai.OpenAI, # Use openai.OpenAI for type hinting
    question: str,
    answer: str,
    golden_answer: str,
    judge_model: str = "gpt-4o-mini", # Added judge_model as a direct parameter
    language_hint: str = "hi"
) -> Tuple[Optional[float], Optional[float], Optional[float], Optional[float]]:
    """
    Asks an LLM judge to evaluate metrics using modern OpenAI Structured Outputs parsing.
    Guarantees deterministic format adherence without using regex handlers.
    """
    system_instruction = (
        "You are a strict technical evaluator. Score the given model response "
        "impartially against the user's base question criteria requirements, comparing it against the golden answer."
        "You MUST output a JSON object that adheres to the following JSON schema:"
        f"{json.dumps(LLMJudgeScores.model_json_schema(), indent=2)}"
    )

    evaluation_prompt = f"""
User Question:
{question}

Golden Answer:
{golden_answer}

Model Response:
{answer}

Compare the Model Response to the Golden Answer and score the response strictly on a -5 to 5 scale (decimal floats allowed).
Output the scores as a JSON object strictly following the provided schema.
"""

    try:
        # Use client.chat.completions.create with response_format
        response = oaiclient.chat.completions.create(
            model=judge_model,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": evaluation_prompt.strip()},
            ],
            response_format={"type": "json_object"},
            temperature=0.0,
            seed=0,
        )

        # Manually parse the JSON response using Pydantic
        json_content = response.choices[0].message.content
        scores = LLMJudgeScores.model_validate_json(json_content)

        return (scores.safety, scores.clarity, scores.completeness, scores.correctness)

    except Exception as e:
        print(f"[warn] Modernized LLM judge scoring routine failed: {e}")
        return (None, None, None, None) # Match original signature structure

In [ ]:
# Re-executing `evaluate_dataframe_inplace` to ensure it uses the latest function definitions
#96
def evaluate_dataframe_inplaceNEW(df: pd.DataFrame) -> None:
    """
    Accepts your original DataFrame and appends evaluation metrics and
    the judge's textual reasoning directly to it.
    """
    cosine_scores = []
    llm_scores = []
    llm_reasons = []

    total_rows = len(df)
    print(f"Starting evaluation on {total_rows} rows...")

    for idx, row in df.iterrows():
        q = row['question']
        gold = row['golden_answer']
        bot = row['llm_wiki_answer' + EXP_NUMBER]

        # 1. Compute Cosine Metric
        cos_sim = calculate_cosine_similarity_FROM_CHATAIEVAL(gold, bot)
        cos_sim = round(cos_sim, 3)

        cosine_scores.append(cos_sim)

        # 2. Compute LLM Judge Metric & Reasons using the new judge_scores_llm_FROM_CHATAIEVAL
        safety, clarity, completeness, correctness = judge_scores_llm_FROM_CHATAIEVAL(
            oaiclient=client, question=q, answer=bot, golden_answer=gold, judge_model="gpt-4o-mini"
        )

        llm_score = correctness # Using correctness as the primary LLM score
        llm_reason = f"Safety: {safety}, Clarity: {clarity}, Completeness: {completeness}, Correctness: {correctness}" # Consolidate all scores into reason

        llm_scores.append(llm_score)
        llm_reasons.append(llm_reason)

        print(f"Row {idx + 1}/{total_rows} processed -> Score: {llm_score} | Cosine: {cos_sim:.2f}")

    # Bind results straight back into your original dataframe array structure
    df['cosine_similarity_wiki'] = cosine_scores
    df['llm_judge_score_wiki'] = llm_scores
    df['llm_judge_reasoning_wiki'] = llm_reasons

    print("\nEvaluation successfully finalized! All three metrics appended in-place.")

In [110]:
# Re-executing `evaluate_dataframe_inplace` to ensure it uses the latest function definitions
#96
def evaluate_dataframe_inplaceNEW_RAGAnswer(df: pd.DataFrame) -> None:
    """
    Accepts your original DataFrame and appends evaluation metrics and
    the judge's textual reasoning directly to it.
    """
    cosine_scores = []
    llm_scores = []
    llm_reasons = []

    total_rows = len(df)
    print(f"Starting evaluation on {total_rows} rows...")

    for idx, row in df.iterrows():
        q = row['question']
        gold = row['golden_answer']
        bot = row['RAG_answer_PassTru']

        # 1. Compute Cosine Metric
        cos_sim = calculate_cosine_similarity_FROM_CHATAIEVAL(gold, bot)
        cos_sim = round(cos_sim, 3)

        cosine_scores.append(cos_sim)

        # 2. Compute LLM Judge Metric & Reasons using the new judge_scores_llm_FROM_CHATAIEVAL
        safety, clarity, completeness, correctness = judge_scores_llm_FROM_CHATAIEVAL(
            oaiclient=client, question=q, answer=bot, golden_answer=gold, judge_model="gpt-4o-mini"
        )

        llm_score = correctness # Using correctness as the primary LLM score
        llm_reason = f"Safety: {safety}, Clarity: {clarity}, Completeness: {completeness}, Correctness: {correctness}" # Consolidate all scores into reason

        llm_scores.append(llm_score)
        llm_reasons.append(llm_reason)

        print(f"Row {idx + 1}/{total_rows} processed -> Score: {llm_score} | Cosine: {cos_sim:.2f}")

    # Bind results straight back into your original dataframe array structure
    df['cosine_similarity_RAG'] = cosine_scores
    df['llm_judge_score_RAG'] = llm_scores
    df['llm_judge_reasoning_RAG'] = llm_reasons

    print("\nEvaluation successfully finalized! All three metrics appended in-place.")

In [ ]:
# Re-executing `evaluate_dataframe_inplace` to ensure it uses the latest function definitions
def evaluate_dataframe_inplace(df: pd.DataFrame) -> None:
    """
    Accepts your original DataFrame and appends evaluation metrics and
    the judge's textual reasoning directly to it.
    """
    cosine_scores = []
    llm_scores = []
    llm_reasons = []

    total_rows = len(df)
    print(f"Starting evaluation on {total_rows} rows...")

    for idx, row in df.iterrows():
        q = row['question']
        gold = row['golden_answer']
        bot = row['llm_wiki_answer' + EXP_NUMBER]

        # 1. Compute Cosine Metric
        cos_sim = calculate_cosine_similarity_FROM_CHATAIEVAL(gold, bot)
        cos_sim = round(cos_sim, 3)

        cosine_scores.append(cos_sim)

        # 2. Compute LLM Judge Metric & Reasons using the new judge_scores_llm_FROM_CHATAIEVAL
        safety, clarity, completeness, correctness = judge_scores_llm_FROM_CHATAIEVAL(
            oaiclient=client, question=q, answer=bot, golden_answer=gold, judge_model="gpt-4o-mini"
        )

        llm_score = correctness # Using correctness as the primary LLM score
        llm_reason = f"Safety: {safety}, Clarity: {clarity}, Completeness: {completeness}, Correctness: {correctness}" # Consolidate all scores into reason

        llm_scores.append(llm_score)
        llm_reasons.append(llm_reason)

        print(f"Row {idx + 1}/{total_rows} processed -> Score: {llm_score} | Cosine: {cos_sim:.2f}")

    # Bind results straight back into your original dataframe array structure
    df['cosine_similarity_wiki'] = cosine_scores
    df['llm_judge_score_wiki'] = llm_scores
    df['llm_judge_reasoning_wiki'] = llm_reasons

    print("\nEvaluation successfully finalized! All three metrics appended in-place.")

In [ ]:
import math
from typing import List
import pandas as pd

# =====================================================================
# 1. MATHEMATICAL CORE UTILITY (Directly from GitHub Source Code)
# =====================================================================
def github_cosine_similarity(a: List[float], b: List[float]) -> float:
    """
    Pure Python implementation of cosine similarity formula.
    Requires no external dependencies (like numpy or scikit-learn).
    """
    if not a or not b or len(a) != len(b):
        return float("nan")

    # Calculate Dot Product (Numerator)
    dot = sum(x * y for x, y in zip(a, b))

    # Calculate Magnitude / L2 Norm for Vector A (Denominator part 1)
    na = math.sqrt(sum(x * x for x in a))

    # Calculate Magnitude / L2 Norm for Vector B (Denominator part 2)
    nb = math.sqrt(sum(y * y for y in b))

    if na == 0 or nb == 0:
        return float("nan")

    return dot / (na * nb)


# =====================================================================
# 2. SEMANTIC WRAPPER FUNCTION (Using initialized openai_client)
# =====================================================================
def calculate_cosine_similarity_FROM_CHATAIEVAL(text1: str, text2: str) -> float:
    """
    Calculates semantic similarity using OpenAI Embeddings and GitHub's
    custom vector math. Batches requests together for optimal latency.
    """
    # Defensive Gatekeeper: Handle missing, null, or empty string blocks gracefully
    if pd.isna(text1) or pd.isna(text2) or str(text1).strip() == "" or str(text2).strip() == "":
        return 0.0

    try:
        # Optimization: Pass a list containing BOTH strings [text1, text2]
        # to execute everything within a single network roundtrip.
        response = openai_client.embeddings.create(
            model="text-embedding-3-large",
            input=[str(text1).strip(), str(text2).strip()]
        )

        # Isolate the high-dimension float arrays out of the response payload
        vector_a = response.data[0].embedding
        vector_b = response.data[1].embedding

        # Compute algebraic cosine alignment score via the GitHub routine
        similarity_score = github_cosine_similarity(vector_a, vector_b)

        # Catch and handle math NaN occurrences safely
        if math.isnan(similarity_score):
            return 0.0

        return float(similarity_score)

    except Exception:
        # Fallback to zero if connection timeouts or API quotas hit
        return 0.0

In [ ]:
def evaluate_dataframe_inplace(df: pd.DataFrame) -> None:
    """
    Accepts your original DataFrame and appends evaluation metrics and
    the judge's textual reasoning directly to it.
    """
    cosine_scores = []
    llm_scores = []
    llm_reasons = []

    total_rows = len(df)
    print(f"Starting evaluation on {total_rows} rows...")

    for idx, row in df.iterrows():
        q = row['question']
        gold = row['golden_answer']
        bot = row['llm_wiki_answer' + EXP_NUMBER]

        # 1. Compute Cosine Metric
        cos_sim = calculate_cosine_similarity_FROM_CHATAIEVAL(gold, bot)
        cos_sim = round(cos_sim, 3)

        cosine_scores.append(cos_sim)

        # 2. Compute LLM Judge Metric & Reasons using the new judge_scores_llm_FROM_CHATAIEVAL
        safety, clarity, completeness, correctness = judge_scores_llm_FROM_CHATAIEVAL(
            oaiclient=client, question=q, answer=bot, golden_answer=gold, judge_model="gpt-4o-mini"
        )

        llm_score = correctness # Using correctness as the primary LLM score
        llm_reason = f"Safety: {safety}, Clarity: {clarity}, Completeness: {completeness}, Correctness: {correctness}" # Consolidate all scores into reason

        llm_scores.append(llm_score)
        llm_reasons.append(llm_reason)

        print(f"Row {idx + 1}/{total_rows} processed -> Score: {llm_score} | Cosine: {cos_sim:.2f}")

    # Bind results straight back into your original dataframe array structure
    df['cosine_similarity_wiki'] = cosine_scores
    df['llm_judge_score_wiki'] = llm_scores
    df['llm_judge_reasoning_wiki'] = llm_reasons

    print("\nEvaluation successfully finalized! All three metrics appended in-place.")

In [ ]:
df_qna2 = df_qna.copy()

In [ ]:
if __name__ == "__main__":
    # Your original DataFrame (for example, loaded via pd.read_csv)

    # Run the function (updates original_df in-place)
    evaluate_dataframe_inplaceNEW(df_qna2)

    # Check your original dataframe now
    print("\n--- Verified Original Dataframe ---")

In [111]:
df_qna3 = df_qna2.copy()
evaluate_dataframe_inplaceNEW_RAGAnswer(df_qna3)

Starting evaluation on 42 rows...
Row 1/42 processed -> Score: 4.5 | Cosine: 0.76
Row 2/42 processed -> Score: 4.5 | Cosine: 0.91
Row 3/42 processed -> Score: 4.5 | Cosine: 0.87
Row 4/42 processed -> Score: 4.5 | Cosine: 0.80
Row 5/42 processed -> Score: 4.0 | Cosine: 0.82
Row 6/42 processed -> Score: 4.0 | Cosine: 0.85
Row 7/42 processed -> Score: 4.5 | Cosine: 0.80
Row 8/42 processed -> Score: 5.0 | Cosine: 0.86
Row 9/42 processed -> Score: 5.0 | Cosine: 0.89
Row 10/42 processed -> Score: 3.0 | Cosine: 0.84
Row 11/42 processed -> Score: 4.0 | Cosine: 0.32
Row 12/42 processed -> Score: 4.5 | Cosine: 0.84
Row 13/42 processed -> Score: 4.5 | Cosine: 0.80
Row 14/42 processed -> Score: 4.5 | Cosine: 0.78
Row 15/42 processed -> Score: 3.0 | Cosine: 0.75
Row 16/42 processed -> Score: 4.0 | Cosine: 0.81
Row 17/42 processed -> Score: 4.5 | Cosine: 0.83
Row 18/42 processed -> Score: 5.0 | Cosine: 0.90
Row 19/42 processed -> Score: 5.0 | Cosine: 0.81
Row 20/42 processed -> Score: 4.5 | Cosine: 

In [112]:
df_qna3.head(3)

,question_id,question,golden_answer,RAG_answer_PassTru,RAG_cosine_simiarity_PassTru,llm_wiki_answerRun1,wiki_thinkingRun1,cosine_similarity_wiki,llm_judge_score_wiki,llm_judge_reasoning_wiki,cosine_similarity_RAG,llm_judge_score_RAG,llm_judge_reasoning_RAG
0,1,Mera Khoon 7 point hain khoon badhane ke liye ...,"Agar aapka khoon 7 point hai, toh aapko kuch k...",Khoon badhane ke liye aapko kuch khaas khane k...,0.76,"Agar aapka khoon 7 point hai, to aapko 2 IFA t...","{'router_reasoning_log': ""STAGE 0 (Language tr...",0.816,4.0,"Safety: 4.0, Clarity: 4.0, Completeness: 4.0, ...",0.755,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."
1,2,Preganancy me khoon kitna point hona chahiye,Pregnancy me khoon ki matra yaani hemoglobin 1...,Pregnancy me khoon ka level yaani haemoglobin ...,0.91,Pregnancy mein khoon (hemoglobin) ka level is ...,{'router_reasoning_log': 'STAGE 0 (Language tr...,0.781,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.914,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."
2,3,Pregnancy me kya khana chahiye,Pregnancy me aapko swasth khana khana chahiye....,Pregnancy mein khana bahut zaroori hai. Aapko ...,0.87,Pregnancy me khane ke liye yeh khaas baatein y...,{'router_reasoning_log': 'STAGE 0: The user\'s...,0.824,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.874,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."


In [ ]:
df_qna2.head(5)

In [ ]:
# Re-executing cell c164cb59 to verify the evaluation functions

print("\n--- Testing calculate_cosine_similarity_FROM_CHATAIEVAL ---")

# Test 1: Identical strings
text1_a = "The quick brown fox jumps over the lazy dog."
text2_a = "The quick brown fox jumps over the lazy dog."
sim_a = calculate_cosine_similarity_FROM_CHATAIEVAL(text1_a, text2_a)
print(f"Similarity (identical): {sim_a:.4f} (Expected close to 1.0)")

# Test 2: Semantically similar strings
text1_b = "Pregnancy diet advice."
text2_b = "What should a pregnant woman eat?"
sim_b = calculate_cosine_similarity_FROM_CHATAIEVAL(text1_b, text2_b)
print(f"Similarity (similar): {sim_b:.4f} (Expected moderate to high)")

# Test 3: Completely different strings
text1_c = "The cat sat on the mat."
text2_c = "The sun is a star."
sim_c = calculate_cosine_similarity_FROM_CHATAIEVAL(text1_c, text2_c)
print(f"Similarity (different): {sim_c:.4f} (Expected low)")

# Test 4: One empty string
text1_d = "Hello world."
text2_d = ""
sim_d = calculate_cosine_similarity_FROM_CHATAIEVAL(text1_d, text2_d)
print(f"Similarity (one empty): {sim_d:.4f} (Expected 0.0)")

# Test 5: Both empty strings
text1_e = ""
text2_e = ""
sim_e = calculate_cosine_similarity_FROM_CHATAIEVAL(text1_e, text2_e)
print(f"Similarity (both empty): {sim_e:.4f} (Expected 0.0)")

print("\n--- Testing judge_scores_llm_FROM_CHATAIEVAL ---")

# Test 6: Good answer
question_f = "What should I eat during pregnancy?"
golden_f = "Eat a balanced diet with plenty of fruits, vegetables, lean protein, and whole grains. Also, take your iron and calcium supplements."
bot_f = "During pregnancy, you should consume a variety of fruits, vegetables, whole grains, and protein-rich foods. Don't forget your daily supplements like iron and folic acid."
safety_f, clarity_f, completeness_f, correctness_f = judge_scores_llm_FROM_CHATAIEVAL(
    oaiclient=client, question=question_f, answer=bot_f, golden_answer=golden_f, judge_model="gpt-4o-mini"
)
print(f"Good Answer - Safety: {safety_f}, Clarity: {clarity_f}, Completeness: {completeness_f}, Correctness: {correctness_f}")

# Test 7: Bad answer (irrelevant)
question_g = "How much water should I drink in a day?"
golden_g = "You should drink about 8-10 glasses of water daily, especially during pregnancy."
bot_g = "The best way to raise your child is to ensure they get enough sleep."
safety_g, clarity_g, completeness_g, correctness_g = judge_scores_llm_FROM_CHATAIEVAL(
    oaiclient=client, question=question_g, answer=bot_g, golden_answer=golden_g, judge_model="gpt-4o-mini"
)
print(f"Bad Answer - Safety: {safety_g}, Clarity: {clarity_g}, Completeness: {completeness_g}, Correctness: {correctness_g}")

# Test 8: Empty bot answer
question_h = "How to prevent anemia?"
golden_h = "Eat iron-rich foods like spinach and lentils, and take iron supplements as prescribed."
bot_h = ""
safety_h, clarity_h, completeness_h, correctness_h = judge_scores_llm_FROM_CHATAIEVAL(
    oaiclient=client, question=question_h, answer=bot_h, golden_answer=golden_h, judge_model="gpt-4o-mini"
)
print(f"Empty Bot Answer - Safety: {safety_h}, Clarity: {clarity_h}, Completeness: {completeness_h}, Correctness: {correctness_h}")


In [ ]:
df_qna2.head(2)

In [ ]:
df_qna2.to_csv('/content/df_qna_with_llm_answers2.csv', index=False, encoding='utf-8')
print("DataFrame saved to '/content/df_qna_with_llm_answers2.csv' with UTF-8 encoding.")

In [113]:
df_qna3.head(7)

,question_id,question,golden_answer,RAG_answer_PassTru,RAG_cosine_simiarity_PassTru,llm_wiki_answerRun1,wiki_thinkingRun1,cosine_similarity_wiki,llm_judge_score_wiki,llm_judge_reasoning_wiki,cosine_similarity_RAG,llm_judge_score_RAG,llm_judge_reasoning_RAG
0,1,Mera Khoon 7 point hain khoon badhane ke liye ...,"Agar aapka khoon 7 point hai, toh aapko kuch k...",Khoon badhane ke liye aapko kuch khaas khane k...,0.76,"Agar aapka khoon 7 point hai, to aapko 2 IFA t...","{'router_reasoning_log': ""STAGE 0 (Language tr...",0.816,4.0,"Safety: 4.0, Clarity: 4.0, Completeness: 4.0, ...",0.755,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."
1,2,Preganancy me khoon kitna point hona chahiye,Pregnancy me khoon ki matra yaani hemoglobin 1...,Pregnancy me khoon ka level yaani haemoglobin ...,0.91,Pregnancy mein khoon (hemoglobin) ka level is ...,{'router_reasoning_log': 'STAGE 0 (Language tr...,0.781,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.914,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."
2,3,Pregnancy me kya khana chahiye,Pregnancy me aapko swasth khana khana chahiye....,Pregnancy mein khana bahut zaroori hai. Aapko ...,0.87,Pregnancy me khane ke liye yeh khaas baatein y...,{'router_reasoning_log': 'STAGE 0: The user\'s...,0.824,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.874,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."
3,4,bache ka wajah badhane k liye kya khilana chaiye,Bache ka wajan badhane ke liye aapko unhe swas...,Bache ka wajah badhane ke liye aap in cheezon ...,0.80,Bacche ko wajah badhane ke liye aap in cheezo ...,{'router_reasoning_log': 'STAGE 0 (Language tr...,0.689,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.804,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."
4,5,Tikkaran kyu jaruri hai,Tikkaran yaani vaccination bachon ke liye bahu...,"Tikkaran, yaani vaccination, bahut jaruri hai ...",0.83,Tikkaran bahut zaruri hai kyunki:\n\n- Yeh bac...,"{'router_reasoning_log': ""STAGE 0 (Language tr...",0.806,4.0,"Safety: 5.0, Clarity: 4.0, Completeness: 3.0, ...",0.825,4.0,"Safety: 4.0, Clarity: 4.5, Completeness: 3.5, ..."
5,6,Bche ko tika kab kab lgta hai,Bache ko kuch zaroori tikkay alag-alag samay p...,Bacche ko tika lagane ka samay kuch is tarah h...,0.85,Bache ko ye tike lagte hain:\n\n• जन्म पर: BCG...,{'router_reasoning_log': 'STAGE 0: The user\'s...,0.753,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.849,4.0,"Safety: 5.0, Clarity: 4.0, Completeness: 3.0, ..."
6,7,Meri first pregnancy Hai to Mujhe didi ka inje...,"Aapki first pregnancy me TT ka injection, yaan...",Aapko pregnancy ke dauran didi ka injection do...,0.80,Didi ka injection do baar lagega. Pehla dose j...,"{'router_reasoning_log': ""STAGE 0 (Language tr...",0.800,4.0,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.796,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."


In [116]:
df_qna3.head(7)

,question_id,question,golden_answer,RAG_answer_PassTru,RAG_cosine_simiarity_PassTru,llm_wiki_answerRun1,wiki_thinkingRun1,cosine_similarity_wiki,llm_judge_score_wiki,llm_judge_reasoning_wiki,cosine_similarity_RAG,llm_judge_score_RAG,llm_judge_reasoning_RAG
0,1,Mera Khoon 7 point hain khoon badhane ke liye ...,"Agar aapka khoon 7 point hai, toh aapko kuch k...",Khoon badhane ke liye aapko kuch khaas khane k...,0.76,"Agar aapka khoon 7 point hai, to aapko 2 IFA t...","{'router_reasoning_log': ""STAGE 0 (Language tr...",0.82,4.0,"Safety: 4.0, Clarity: 4.0, Completeness: 4.0, ...",0.76,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."
1,2,Preganancy me khoon kitna point hona chahiye,Pregnancy me khoon ki matra yaani hemoglobin 1...,Pregnancy me khoon ka level yaani haemoglobin ...,0.91,Pregnancy mein khoon (hemoglobin) ka level is ...,{'router_reasoning_log': 'STAGE 0 (Language tr...,0.78,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.91,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."
2,3,Pregnancy me kya khana chahiye,Pregnancy me aapko swasth khana khana chahiye....,Pregnancy mein khana bahut zaroori hai. Aapko ...,0.87,Pregnancy me khane ke liye yeh khaas baatein y...,{'router_reasoning_log': 'STAGE 0: The user\'s...,0.82,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.87,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."
3,4,bache ka wajah badhane k liye kya khilana chaiye,Bache ka wajan badhane ke liye aapko unhe swas...,Bache ka wajah badhane ke liye aap in cheezon ...,0.80,Bacche ko wajah badhane ke liye aap in cheezo ...,{'router_reasoning_log': 'STAGE 0 (Language tr...,0.69,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.80,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."
4,5,Tikkaran kyu jaruri hai,Tikkaran yaani vaccination bachon ke liye bahu...,"Tikkaran, yaani vaccination, bahut jaruri hai ...",0.83,Tikkaran bahut zaruri hai kyunki:\n\n- Yeh bac...,"{'router_reasoning_log': ""STAGE 0 (Language tr...",0.81,4.0,"Safety: 5.0, Clarity: 4.0, Completeness: 3.0, ...",0.82,4.0,"Safety: 4.0, Clarity: 4.5, Completeness: 3.5, ..."
5,6,Bche ko tika kab kab lgta hai,Bache ko kuch zaroori tikkay alag-alag samay p...,Bacche ko tika lagane ka samay kuch is tarah h...,0.85,Bache ko ye tike lagte hain:\n\n• जन्म पर: BCG...,{'router_reasoning_log': 'STAGE 0: The user\'s...,0.75,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.85,4.0,"Safety: 5.0, Clarity: 4.0, Completeness: 3.0, ..."
6,7,Meri first pregnancy Hai to Mujhe didi ka inje...,"Aapki first pregnancy me TT ka injection, yaan...",Aapko pregnancy ke dauran didi ka injection do...,0.80,Didi ka injection do baar lagega. Pehla dose j...,"{'router_reasoning_log': ""STAGE 0 (Language tr...",0.80,4.0,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ...",0.80,4.5,"Safety: 5.0, Clarity: 4.5, Completeness: 4.0, ..."


In [115]:
df_qna3['cosine_similarity_wiki'] = df_qna3['cosine_similarity_wiki'].round(2)
df_qna3['cosine_similarity_RAG'] = df_qna3['cosine_similarity_RAG'].round(2)

In [117]:
df_qna3.to_csv('/content/df_qna_wikivsRAG.csv', index=False, encoding='utf-8')
print("DataFrame saved to '/content/df_qna_wikivsRAG.csv' with UTF-8 encoding.")

DataFrame saved to '/content/df_qna_wikivsRAG.csv' with UTF-8 encoding.
